In [1]:
import pandas as pd

df = pd.read_csv('uncleaned_data.csv')

In [2]:
import pandas as pd
import numpy as np

# ==========================
# 1. Load raw data
# ==========================

df = pd.read_csv("uncleaned_data.csv")
df_clean = df.copy()

print("Raw shape:", df_clean.shape)

# ==========================
# 2. Basic helpers
# ==========================

def safe_lower(col):
    """Lowercase a text column safely."""
    return col.fillna("").astype(str).str.lower()

# ==========================
# 3. Numeric + binary cleanup
# ==========================

# Columns you *expect* to be numeric (coerce errors to NaN)
numeric_like_cols = [
    "Age",
    "BMI",
    "ALIF Count",
    "Lateral Count",
    "Average PI",
    "PI-LL angle mismatch",
    "ABS PI-LL angle mismatch",
    "Post-op SS",
    "post PI",
    "post PT",
    "post LL",
    "post SVA",
    "length of hospital stay (d)",
]

for col in numeric_like_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Binary / indicator columns that should be 0/1
binary_like_cols = [
    "prior back surgeries? (y=1)",
    "Perc screws?",
    "Open",
    "Open Check V2",
    "Standalone XLIF Check",
    "Retroperitoneal Approach (LLIF ± ALIF)",
    "Anterior + Posterior Apporoach",
    "Osteotomies (yes/no)",
    "ACR (y=1)",
    "PI-LL Mismatch Category (1 = mismatch > +/- 9",
    "PI-LL Mismatch Category (1 = mismatch > +/- 10",
    "(1 = PI>50)",
    "infection 1=yes",
    "DVT  1=yes",
    "PE  1=yes",
    "MI 1=yes",
    "femoral palsy (knee extension weakness) 1=yes",
    "hip flexion weakness (iliopsoas weakness)  1=yes",
    "acute thigh paresthesia (immediate post op)",
    "psoas hematoma",
    # fusion level indicators:
    "T12-L1",
    "L1-L2",
    "L2-L3",
    "L3-L4",
    "L4-L5",
    "L5-S1",
]

for col in binary_like_cols:
    if col in df_clean.columns:
        ser = df_clean[col]
        # First normalize text yes/no if present
        if ser.dtype == "object":
            ser = safe_lower(ser)
            ser = ser.replace(
                {
                    "yes": 1, "y": 1, "true": 1, "t": 1,
                    "no": 0, "n": 0, "false": 0, "f": 0,
                }
            )
        df_clean[col] = pd.to_numeric(ser, errors="coerce").fillna(0).astype(int)

# Optional: simple sex coding if Sex is present
if "Sex" in df_clean.columns:
    # Map common patterns to 0/1, keep others as-is
    sex_ser = safe_lower(df_clean["Sex"])
    sex_map = {
        "m": 1, "male": 1,
        "f": 0, "female": 0,
    }
    df_clean["Sex"] = sex_ser.map(sex_map).fillna(sex_ser)  # leaves unknowns as text

# ==========================
# 4. Surgery feature engineering
# ==========================

case_col = "Case/Type of Surgery"
level_cols = ["T12-L1", "L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1"]

# Ensure level columns exist and are 0/1 ints
for col in level_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0).astype(int)
    else:
        # create if missing to avoid KeyErrors later
        df_clean[col] = 0

# 4a. Number of fused levels
df_clean["levels_fused_count"] = df_clean[level_cols].sum(axis=1)

# 4b. Construct span (T12-L1=0, ..., L5-S1=5)
level_index_map = {lvl: i for i, lvl in enumerate(level_cols)}

def compute_span(row):
    fused_idxs = [level_index_map[col] for col in level_cols if row[col] == 1]
    if not fused_idxs:
        return 0
    return max(fused_idxs) - min(fused_idxs) + 1

df_clean["construct_span_levels"] = df_clean.apply(compute_span, axis=1)

# 4c. Region flags
df_clean["thoracolumbar_junction"] = (
    (df_clean["T12-L1"] == 1) | (df_clean["L1-L2"] == 1)
).astype(int)

df_clean["upper_lumbar"] = (
    (df_clean["L1-L2"] == 1) | (df_clean["L2-L3"] == 1)
).astype(int)

df_clean["lower_lumbar"] = (df_clean["L4-L5"] == 1).astype(int)
df_clean["lumbosacral"] = (df_clean["L5-S1"] == 1).astype(int)

# 4d. Revision / deformity flags + LLIF/XLIF/ALIF from case text
if case_col in df_clean.columns:
    df_clean[case_col] = safe_lower(df_clean[case_col])

    # revision flag
    df_clean["revision_surgery"] = df_clean[case_col].str.contains(
        "removal of hardware|revision", na=False
    ).astype(int)

    # deformity-related wording
    df_clean["deformity_case_text"] = df_clean[case_col].str.contains(
        "deformity|sagittal imbalance|flat back|scoliosis", na=False
    ).astype(int)

    # NEW: LLIF / "lateral" flag (some cases only say "lateral")
    df_clean["llif_or_lateral_text"] = df_clean[case_col].str.contains(
        r"\bllif\b|\blateral\b", na=False
    ).astype(int)

    # NEW: XLIF flag
    df_clean["xlif_text"] = df_clean[case_col].str.contains(
        r"\bxlif\b", na=False
    ).astype(int)

    # NEW: ALIF flag
    df_clean["alif_text"] = df_clean[case_col].str.contains(
        r"\balif\b", na=False
    ).astype(int)

else:
    df_clean["revision_surgery"] = 0
    df_clean["deformity_case_text"] = 0
    df_clean["llif_or_lateral_text"] = 0
    df_clean["xlif_text"] = 0
    df_clean["alif_text"] = 0

# ==========================
# 5. Pre-op diagnosis feature engineering
# ==========================

dx_col = "Pre op Diagnosis (back pain, adjacent segment disease, spondy)"

if dx_col in df_clean.columns:
    df_clean[dx_col] = safe_lower(df_clean[dx_col])

    df_clean["dx_adjacent_segment"] = df_clean[dx_col].str.contains(
        "adjacent segment", na=False
    ).astype(int)

    df_clean["dx_spondylolisthesis"] = df_clean[dx_col].str.contains(
        "spondylolisthesis", na=False
    ).astype(int)

    df_clean["dx_spondylosis"] = df_clean[dx_col].str.contains(
        "spondylosis", na=False
    ).astype(int)

    df_clean["dx_stenosis"] = df_clean[dx_col].str.contains(
        "stenosis", na=False
    ).astype(int)

    df_clean["dx_scoliosis"] = df_clean[dx_col].str.contains(
        "scoliosis", na=False
    ).astype(int)

    df_clean["dx_flat_back"] = df_clean[dx_col].str.contains(
        "flat back", na=False
    ).astype(int)

    df_clean["dx_sagittal_imbalance"] = df_clean[dx_col].str.contains(
        "sagittal imbalance", na=False
    ).astype(int)

    df_clean["dx_post_laminectomy"] = df_clean[dx_col].str.contains(
        "post-laminectomy", na=False
    ).astype(int)

    df_clean["dx_deformity"] = (
        df_clean["dx_scoliosis"]
        | df_clean["dx_flat_back"]
        | df_clean["dx_sagittal_imbalance"]
        | df_clean[dx_col].str.contains("deformity", na=False)
    ).astype(int)
else:
    # If column missing, create zeros so later modeling code doesn't break
    for col in [
        "dx_adjacent_segment",
        "dx_spondylolisthesis",
        "dx_spondylosis",
        "dx_stenosis",
        "dx_scoliosis",
        "dx_flat_back",
        "dx_sagittal_imbalance",
        "dx_post_laminectomy",
        "dx_deformity",
    ]:
        df_clean[col] = 0

# ==========================
# 6. Save cleaned CSV
# ==========================

out_path = "cleaned_data.csv"
df_clean.to_csv(out_path, index=False)

print("Cleaned data saved to:", out_path)
print("Cleaned shape:", df_clean.shape)


Raw shape: (546, 96)
Cleaned data saved to: cleaned_data.csv
Cleaned shape: (546, 116)
